In [17]:
# US CENSUS DATA ANALYSIS PROJECT
# Population and Economy Trend Analysis Across US States/Counties

# Install Required Packages
# pip install pandas requests sqlalchemy psycopg2-binary matplotlib seaborn

import requests
import pandas as pd
from sqlalchemy import create_engine

# STEP 1: DEFINE YOUR CENSUS API KEY

# Get a free API key from:
# https://api.census.gov/data/key_signup.html

API_KEY = "0fbb94d03c9fa7a80b770fd1311fa8ed4c7d8717"

# # STEP 2: DEFINE DATA VARIABLES REQUIRED FOR THE PROJECT

# American Community Survey (ACS) 5-Year Data
# Dataset Documentation:
# https://api.census.gov/data.html

# Variables selected for population and economic analysis

variables = {
    "NAME": "County/State Name",
    "B01003_001E": "Population",
    "B19013_001E": "Median_Household_Income",
    "B17001_002E": "People_Below_Poverty",
    "B23025_003E": "Employed_Population",
    "B23025_005E": "Unemployed_Population",
    "B15003_022E": "Bachelors_Degree",
    "B25077_001E": "Median_Home_Value"
}

# Convert variable list into Census API format
variable_string = ",".join(variables.keys())

# STEP 3: CREATE API REQUEST URL

# Pull county-level data for all counties in the USA

url = (
    f"https://api.census.gov/data/2023/acs/acs5?"
    f"get={variable_string}"
    f"&for=county:*"
    f"&in=state:*"
    f"&key={API_KEY}"
)

print("Requesting data from Census API...")
response = requests.get(url)

Requesting data from Census API...


In [18]:
# STEP 4: CONVERT API RESPONSE TO DATAFRAME

if response.status_code == 200:

    data = response.json()

    # First row contains column names
    df = pd.DataFrame(data[1:], columns=data[0])

    print("Data successfully downloaded.")

else:
    print("Error:", response.status_code)

Data successfully downloaded.


In [19]:
# STEP 5: RENAME COLUMNS
df.rename(columns=variables, inplace=True)

In [20]:
# STEP 6: CONVERT NUMERIC COLUMNS

numeric_columns = [
    "Population",
    "Median_Household_Income",
    "People_Below_Poverty",
    "Employed_Population",
    "Unemployed_Population",
    "Bachelors_Degree",
    "Median_Home_Value"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [21]:
# STEP 9: SAVE DATA TO CSV

df.to_csv("us_census_county_data_raw.csv", index=False)

print("\nCSV file saved successfully.")


CSV file saved successfully.


In [22]:
# STEP 7: CREATE NEW ANALYTICAL VARIABLES

# Poverty Rate
df["Poverty_Rate"] = (
    df["People_Below_Poverty"] / df["Population"]
) * 100

# Employment Rate
df["Employment_Rate"] = (
    df["Employed_Population"] /
    (df["Employed_Population"] + df["Unemployed_Population"])
) * 100

# Unemployment Rate
df["Unemployment_Rate"] = (
    df["Unemployed_Population"] /
    (df["Employed_Population"] + df["Unemployed_Population"])
) * 100

# Education Rate
df["Education_Rate"] = (
    df["Bachelors_Degree"] / df["Population"]
) * 100

In [23]:
# STEP 8: DISPLAY SAMPLE DATA

print("\nSample Dataset:")
print(df.head())


Sample Dataset:
         County/State Name  Population  Median_Household_Income  \
0  Autauga County, Alabama       59285                    69841   
1  Baldwin County, Alabama      239945                    75019   
2  Barbour County, Alabama       24757                    44290   
3     Bibb County, Alabama       22152                    51215   
4   Blount County, Alabama       59292                    61096   

   People_Below_Poverty  Employed_Population  Unemployed_Population  \
0                  6275                27070                    688   
1                 24819               113171                   3615   
2                  4746                 9074                    518   
3                  4258                 9375                    936   
4                  8269                27180                   1586   

   Bachelors_Degree  Median_Home_Value state county  Poverty_Rate  \
0              6518             197900    01    001     10.584465   
1             3

In [24]:
# STEP 9: SAVE DATA TO CSV

df.to_csv("us_census_county_data.csv", index=False)

print("\nCSV file saved successfully.")


CSV file saved successfully.
